## **Project 1: Summarize News Articles with ML**

**Algorithm type:** Abstractive Summarization  
**Dataset:** Kaggle News Summarization Dataset/ https://www.kaggle.com/datasets/sbhatti/news-summarization/data  


## **1. Install Required Libraries**


In [30]:
!pip install -q kaggle transformers datasets evaluate rouge_score sentencepiece accelerate scikit-learn pandas numpy

In [16]:
!pip install -q evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00


## **2. Import Libraries**

In [17]:
import os
import glob
import shutil
import inspect
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
GPU name: Tesla T4


## **3. Connect Kaggle Dataset with Google Colab using Kaggle API**



In [4]:
KAGGLE_JSON_PATH = "/content/kaggle.json"
KAGGLE_DIR = os.path.expanduser("~/.kaggle")
KAGGLE_CONFIG_PATH = os.path.join(KAGGLE_DIR, "kaggle.json")

os.makedirs(KAGGLE_DIR, exist_ok=True)

if not os.path.exists(KAGGLE_CONFIG_PATH):
    try:
        from google.colab import files
        print("Please upload your kaggle.json file now.")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError("kaggle.json was not uploaded. Please upload the correct Kaggle API token file.")
        shutil.copy(KAGGLE_JSON_PATH, KAGGLE_CONFIG_PATH)
        os.chmod(KAGGLE_CONFIG_PATH, 0o600)
        print("Kaggle API token configured successfully.")
    except Exception as e:
        raise RuntimeError(
            "Kaggle setup failed. Make sure you are running this notebook in Google Colab "
            "and upload a valid kaggle.json file. Original error: " + str(e)
        )
else:
    os.chmod(KAGGLE_CONFIG_PATH, 0o600)
    print("Kaggle API token already exists and is configured.")

Please upload your kaggle.json file now.


Saving kaggle.json to kaggle.json
Kaggle API token configured successfully.


## 4. Download Dataset from Kaggle


Dataset link: `https://www.kaggle.com/datasets/sbhatti/news-summarization`

In [5]:
DATASET_NAME = "sbhatti/news-summarization"
DATA_DIR = "/content/news_summarization_dataset"

os.makedirs(DATA_DIR, exist_ok=True)

!kaggle datasets download -d {DATASET_NAME} -p {DATA_DIR} --unzip

csv_files = glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)

if not csv_files:
    raise FileNotFoundError("No CSV file found after downloading the Kaggle dataset.")

csv_path = csv_files[0]
print("CSV file found:", csv_path)

Dataset URL: https://www.kaggle.com/datasets/sbhatti/news-summarization
License(s): CC0-1.0
100% 1.37G/1.37G [00:19<00:00, 73.6MB/s]

CSV file found: /content/news_summarization_dataset/data.csv


## **5. Load First 400 Rows Only**



In [6]:
N_ROWS = 400

raw_df = pd.read_csv(csv_path, nrows=N_ROWS, on_bad_lines="skip")

print("Dataset shape:", raw_df.shape)
raw_df.head()

Dataset shape: (400, 5)


,Unnamed: 0,ID,Content,Summary,Dataset
0,0,f49ee725a0360aa6881ed1f7999cc531885dd06a,New York police are concerned drones could bec...,Police have investigated criminals who have ri...,CNN/Daily Mail
1,1,808fe317a53fbd3130c9b7563341a7eea6d15e94,By . Ryan Lipman . Perhaps Australian porn sta...,Porn star Angela White secretly filmed sex act...,CNN/Daily Mail
2,2,98fd67bd343e58bc4e275bbb5a4ea454ec827c0d,"This was, Sergio Garcia conceded, much like be...",American draws inspiration from fellow country...,CNN/Daily Mail
3,3,e12b5bd7056287049d9ec98e41dbb287bd19a981,An Ebola outbreak that began in Guinea four mo...,World Health Organisation: 635 infections and ...,CNN/Daily Mail
4,4,b83e8bcfcd51419849160e789b6658b21a9aedcd,By . Associated Press and Daily Mail Reporter ...,A sinkhole opened up at 5:15am this morning in...,CNN/Daily Mail


## **6. Exploratory Data Analysis-EDA**



In [7]:
print("Columns in dataset:")
print(raw_df.columns.tolist())

print("\nDataset information:")
raw_df.info()

Columns in dataset:
['Unnamed: 0', 'ID', 'Content', 'Summary', 'Dataset']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  400 non-null    int64 
 1   ID          374 non-null    object
 2   Content     400 non-null    object
 3   Summary     400 non-null    object
 4   Dataset     400 non-null    object
dtypes: int64(1), object(4)
memory usage: 15.8+ KB


In [8]:
print("Missing values per column:")
print(raw_df.isnull().sum())

print("\nDuplicate rows:", raw_df.duplicated().sum())

Missing values per column:
Unnamed: 0     0
ID            26
Content        0
Summary        0
Dataset        0
dtype: int64

Duplicate rows: 0


### **Automatically Detect Article and Summary Columns**


In [9]:
def detect_article_summary_columns(df):
    article_keywords = ["article", "text", "document", "content", "story", "news", "body"]
    summary_keywords = ["summary", "summaries", "highlight", "highlights", "abstract", "target", "headline"]

    columns = list(df.columns)
    lower_map = {col: str(col).lower().strip() for col in columns}

    article_col = None
    summary_col = None

    for col, lower_col in lower_map.items():
        if any(keyword in lower_col for keyword in article_keywords):
            article_col = col
            break

    for col, lower_col in lower_map.items():
        if any(keyword in lower_col for keyword in summary_keywords):
            summary_col = col
            break

    object_cols = df.select_dtypes(include=["object"]).columns.tolist()

    if article_col is None or summary_col is None or article_col == summary_col:
        if len(object_cols) < 2:
            raise ValueError("Could not detect article and summary columns. The dataset must contain at least two text columns.")

        avg_lengths = {}
        for col in object_cols:
            avg_lengths[col] = df[col].astype(str).str.len().mean()

        sorted_cols = sorted(avg_lengths, key=avg_lengths.get, reverse=True)

        if article_col is None:
            article_col = sorted_cols[0]

        if summary_col is None or summary_col == article_col:
            for col in sorted_cols[1:]:
                if col != article_col:
                    summary_col = col
                    break

    return article_col, summary_col

article_col, summary_col = detect_article_summary_columns(raw_df)

print("Detected article column:", article_col)
print("Detected summary column:", summary_col)

Detected article column: Content
Detected summary column: Summary


In [10]:
data = raw_df[[article_col, summary_col]].copy()
data.columns = ["article", "summary"]

data["article_length"] = data["article"].astype(str).str.len()
data["summary_length"] = data["summary"].astype(str).str.len()

data[["article", "summary", "article_length", "summary_length"]].head()

,article,summary,article_length,summary_length
0,New York police are concerned drones could bec...,Police have investigated criminals who have ri...,2906,289
1,By . Ryan Lipman . Perhaps Australian porn sta...,Porn star Angela White secretly filmed sex act...,1612,360
2,"This was, Sergio Garcia conceded, much like be...",American draws inspiration from fellow country...,5515,177
3,An Ebola outbreak that began in Guinea four mo...,World Health Organisation: 635 infections and ...,3666,367
4,By . Associated Press and Daily Mail Reporter ...,A sinkhole opened up at 5:15am this morning in...,3255,346


In [11]:
print("Article length statistics:")
print(data["article_length"].describe())

print("\nSummary length statistics:")
print(data["summary_length"].describe())

Article length statistics:
count      400.000000
mean      3994.107500
std       3737.722342
min        254.000000
25%       2228.500000
50%       3347.000000
75%       5076.750000
max      51474.000000
Name: article_length, dtype: float64

Summary length statistics:
count     400.000000
mean      319.377500
std       303.313473
min        55.000000
25%       146.000000
50%       251.500000
75%       343.500000
max      1853.000000
Name: summary_length, dtype: float64


## **7. Data Cleaning**


In [12]:
def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = " ".join(text.split())
    return text.strip()

data = data[["article", "summary"]].copy()

data["article"] = data["article"].apply(clean_text)
data["summary"] = data["summary"].apply(clean_text)

data = data.dropna()
data = data.drop_duplicates()

data = data[
    (data["article"].str.len() >= 100) &
    (data["summary"].str.len() >= 20)
].reset_index(drop=True)

print("Cleaned dataset shape:", data.shape)
data.head()

Cleaned dataset shape: (400, 2)


,article,summary
0,New York police are concerned drones could bec...,Police have investigated criminals who have ri...
1,By . Ryan Lipman . Perhaps Australian porn sta...,Porn star Angela White secretly filmed sex act...
2,"This was, Sergio Garcia conceded, much like be...",American draws inspiration from fellow country...
3,An Ebola outbreak that began in Guinea four mo...,World Health Organisation: 635 infections and ...
4,By . Associated Press and Daily Mail Reporter ...,A sinkhole opened up at 5:15am this morning in...


## **8. Train-Test Split**

Use 80% of the data for training and 20% for testing.

In [18]:
if len(data) < 10:
    raise ValueError("Not enough valid rows after cleaning. Please increase N_ROWS or check column detection.")

train_df, test_df = train_test_split(
    data,
    test_size=0.2,
    random_state=SEED,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

Training rows: 320
Testing rows: 80


## **9. Tokenization**


In [19]:
MODEL_NAME = "t5-small"
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['article', 'summary'],
    num_rows: 320
})
Dataset({
    features: ['article', 'summary'],
    num_rows: 80
})


## **10. Load Abstractive Summarization Model**



In [20]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded:", MODEL_NAME)
print("Device:", device)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded: t5-small
Device: cuda


### **Apply Tokenization to Dataset**

In [21]:
def preprocess_function(examples):
    inputs = ["summarize: " + article for article in examples["article"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    try:
        labels = tokenizer(
            text_target=examples["summary"],
            max_length=MAX_TARGET_LENGTH,
            truncation=True
        )
    except TypeError:
        # Compatibility fallback for older tokenizer versions
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(
                examples["summary"],
                max_length=MAX_TARGET_LENGTH,
                truncation=True
            )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

print(tokenized_train)
print(tokenized_test)

Map:   0%|          | 0/320 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 320
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 80
})


## **11. Fine-Tune T5-small**



In [23]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args_params = dict(
    output_dir="/content/news_summary_model",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    logging_steps=20,
    save_strategy="epoch"
)

signature = inspect.signature(Seq2SeqTrainingArguments.__init__)

if "eval_strategy" in signature.parameters:
    training_args_params["eval_strategy"] = "epoch"
elif "evaluation_strategy" in signature.parameters:
    training_args_params["evaluation_strategy"] = "epoch"

training_args = Seq2SeqTrainingArguments(**training_args_params)

trainer_params = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator
)

trainer_signature = inspect.signature(Seq2SeqTrainer.__init__)

if "processing_class" in trainer_signature.parameters:
    trainer_params["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature.parameters:
    trainer_params["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_params)

trainer.train()

Epoch,Training Loss,Validation Loss
1,2.393114,2.703124


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=160, training_loss=2.8917231798171996, metrics={'train_runtime': 29.3237, 'train_samples_per_second': 10.913, 'train_steps_per_second': 5.456, 'total_flos': 42640597647360.0, 'train_loss': 2.8917231798171996, 'epoch': 1.0})

In [24]:
trainer.save_model("/content/final_t5_news_summarizer")
tokenizer.save_pretrained("/content/final_t5_news_summarizer")

print("Model saved at: /content/final_t5_news_summarizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved at: /content/final_t5_news_summarizer


## **12. Generate Summary**

This function generates an abstractive summary from any news article.

In [25]:
def generate_summary(article, max_input_length=512, max_summary_length=128, min_summary_length=20):
    article = clean_text(article)
    input_text = "summarize: " + article

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=max_input_length,
        truncation=True
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    model.eval()
    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs.get("attention_mask"),
            max_length=max_summary_length,
            min_length=min_summary_length,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [26]:
sample_article = test_df.iloc[0]["article"]
actual_summary = test_df.iloc[0]["summary"]
predicted_summary = generate_summary(sample_article)

print("Original Article:\n")
print(sample_article[:1500])

print("\nActual Summary:\n")
print(actual_summary)

print("\nPredicted Summary:\n")
print(predicted_summary)

Original Article:

Monaco coach Leonardo Jardim warned his players not to rest on their laurels, telling them that they face the most difficult night of their careers when they defend their 3-1 lead at Stade Louis II against Arsenal on Tuesday night. Portuguese coach Jardim emphasised the threat of Mesut Ozil and Santi Cazorla and is trying to prepare his players for an expected attacking onslaught from Arsenal, with the Barclays Premier League side needing to at least three goals to go through in the Champions League tie. Jardim said: 'At this stage we're now at half-time and we're two goals ahead but it's true that our opponents have so many quality players in their squad. So it will be very difficult for us and everyone knows if it will be the most difficult game we've had in years. Monaco coach Leonardo Jardim speaks during his Champions League press conference on Monday . Jardim feels Arsenal still pose a threat even if Monaco carry a 3-1 lead from their first leg win at the Emira

## **13. Evaluate with ROUGE Score**

ROUGE is commonly used to evaluate text summarization. It compares the generated summary with the original human-written summary.

In [27]:
rouge = evaluate.load("rouge")

predictions = []
references = []

EVAL_SAMPLES = min(20, len(test_df))

for i in range(EVAL_SAMPLES):
    article = test_df.iloc[i]["article"]
    reference_summary = test_df.iloc[i]["summary"]
    generated_summary = generate_summary(article)

    predictions.append(generated_summary)
    references.append(reference_summary)

rouge_results = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True
)

print("ROUGE Evaluation Results:")
for key, value in rouge_results.items():
    print(f"{key}: {value:.4f}")

ROUGE Evaluation Results:
rouge1: 0.2844
rouge2: 0.0643
rougeL: 0.1762
rougeLsum: 0.1744


In [28]:
results_df = pd.DataFrame({
    "Article": test_df.iloc[:EVAL_SAMPLES]["article"].values,
    "Actual Summary": references,
    "Predicted Summary": predictions
})

results_df.head()

,Article,Actual Summary,Predicted Summary
0,Monaco coach Leonardo Jardim warned his player...,Monaco beat Arsenal 3-1 at the Emirates in Feb...,Monaco coach Leonardo Jardim warned his player...
1,"(CNN) -- In a bookstore, I saw a woman taking ...","Bob Greene: A new practice called' ""showroomin...","""Showrooming"" is a relatively new phenomenon a..."
2,Russia's economic development ministry estimat...,The Russian government has warned the economy ...,Russia's economic development ministry estimat...
3,"Paramedics treated about 12,000 people who wer...",Ambulances attend more than 60 incidents on av...,Scottish ambulance service said alcohol had a ...
4,He claimed there were few vessels to enforce n...,"The UK will be a ""laughing stock"" in Europe if...","minister Lord Gardiner says he is ""stunned"" by..."


## **14. Test on Custom News Article**



In [29]:
custom_article = "Pakistan's technology sector has seen significant growth in recent years as more startups enter fields such as artificial intelligence, fintech, e-commerce, and software development. Industry experts believe that young developers, entrepreneurs, and freelancers can play an important role in improving the country's digital economy. They also suggest that universities and training institutes should focus on practical skills, real-world projects, and industry collaboration to prepare students for modern technology jobs."

custom_summary = generate_summary(custom_article)

print("Custom Article Summary:\n")
print(custom_summary)

Custom Article Summary:

Pakistan's technology sector has seen significant growth in recent years. experts believe young developers, entrepreneurs, and freelancers can play an important role in improving the country's digital economy. universities and training institutes should focus on practical skills, real-world projects and industry collaboration to prepare students for modern technology jobs.
